# 三维道路 DAS + 锤击地下空洞探测算法原型

本笔记用于解释当前 `main.py` 演示的第一阶段算法。当前模型把空洞简化为一个点绕射体，因此只能用于研究三维几何关系、走时关系和定位目标函数，不能代表完整三维弹性波传播。


## 1. 项目背景

道路地下空洞、脱空和松散区会改变浅层地震波传播。项目目标是利用道路一侧既有或布设的 DAS 光纤，以及道路另一侧的锤击震源，形成多炮多道观测系统，对地下浅层空洞进行三维定位。

第一阶段不追求工业级正演，而是先保证三维坐标、DAS 光纤采样、运动学绕射正演、三维定位搜索和不确定性诊断形成可运行闭环。


## 2. 三维坐标系统

坐标约定如下：

- `x`：沿道路或沿 DAS 光纤方向；
- `y`：横穿道路方向；
- `depth`：向下为正，单位为 m。

所有几何对象都必须是三维：`source_xyz`、`receiver_xyz`、`receiver_polyline` 和 `void_xyz` 都包含 `x, y, depth`。这避免把二维剖面算法误包装为三维算法。


## 3. 道路、光纤、锤击点和空洞几何

当前默认场景：

- 道路长度 `80 m`，宽度 `15 m`；
- DAS 光纤位于道路一侧 `y=0 m`，沿 `x=0..80 m` 延伸；
- 锤击点位于另一侧 `y=15 m`，沿 `x=0..80 m` 排列；
- 空洞位于道路中部浅层 `void_xyz=(40.0, 7.5, 2.0)`。

因此光纤、锤击点和空洞共同构成真实三维几何，空洞不在光纤正下方。


In [ ]:
from pathlib import Path
import sys

CODE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from main import build_stage1_scene

scene = build_stage1_scene()
scene['road_length_m'], scene['road_width_m'], scene['void_model'].void_xyz.xyz


## 4. 三维运动学绕射正演原理

当前把空洞简化为一个点绕射体。对任意震源 `s`、接收点 `r` 和空洞候选点 `p`，总走时为：

```text
总走时 = 震源到空洞的走时 + 空洞到接收点的走时
t_total = |s - p| / v + |p - r| / v
```

这里的距离是三维欧氏距离，所以 `y` 方向横向偏移会直接影响走时。


## 5. 合成记录生成方法

`simulate_kinematic_diffraction` 对每个 source-receiver 对计算理论绕射走时，并在该时刻放置一个 Ricker 子波。输出数据维度为：

```text
n_sources x n_receivers x n_times
```

可选的几何扩散只是一种简单振幅衰减，不代表真实弹性散射振幅。


## 6. 三维 x-y-depth 定位搜索

定位函数 `travel_time_energy_stack` 遍历 `search_x`、`search_y`、`search_depth`。每个候选点都会生成一张理论走时表，然后从记录对应时间附近提取绝对振幅并叠加。

目标函数高值表示该候选点更能解释多炮多道记录中的绕射能量。输出不是二维剖面，而是完整三维目标函数体。


## 7. DAS 光纤观测的当前近似

当前 `ReceiverPolyline3D` 只把光纤折线按固定通道间距采样成点接收器，并记录切向量和 `gauge_length_m` metadata。

这不是严格 DAS 轴向应变。真实 DAS 需要弹性位移场 `u`、应变张量 `epsilon(u)`、光纤切向量 `e`，以及 `e^T epsilon(u) e` 的 gauge-length 平均。


## 8. 定位不确定性和 y-depth 混淆

当前不确定性指标来自目标函数体：

- best/second ratio：最佳点是否唯一；
- half-max 区域：接近最佳得分的候选体积有多宽；
- x/y/depth 半高宽：不同方向上的定位扩散；
- y-depth confusion：横向位置和埋深是否难以区分。

单侧 DAS 光纤和对侧锤击几何可能使 `y` 与 `depth` 的 trade-off 比较明显，这是后续需要重点分析的问题。


In [ ]:
from main import run

summary = run(output_dir=CODE_DIR / 'outputs', quiet=True)
summary['true_xyz'], summary['best_xyz'], summary['localization_error_m'], summary['uncertainty']['y_depth_confusion']


运行 `main.py` 后可查看以下图像：

- `code/outputs/geometry_3d.png`：三维道路 DAS + 锤击几何；
- `code/outputs/synthetic_gather.png`：单炮合成记录；
- `code/outputs/localization_slices.png`：定位目标函数切片。


## 9. 当前算法限制

当前阶段的限制：

- 空洞是单点绕射体，不是有限尺寸真实空洞；
- 速度模型是常速度近似；
- 没有求解三维弹性波方程；
- 没有自由表面、衰减、各向异性或模式转换；
- DAS 只是点通道近似，不是真实 gauge-length 轴向应变；
- 定位目标函数是 travel-time energy stack，不是 FWI 或概率反演。


## 10. 后续接入 Devito 或 OpenSWPC 的方向

后续可以保持当前三维几何 API 不变，把正演后端替换为：

- Devito：Python 原生三维 acoustic/elastic 有限差分原型；
- OpenSWPC：成熟 MIT 许可证三维弹性/黏弹性外部求解器。

在接入高保真后端前，应先完成 DAS gauge-length 轴向应变观测算子，并持续验证 source、receiver、void 和 `x-y-depth` 搜索几何一致。
